# Notebook 02: Performance Benchmark, Metric Analysis & Confusion Matrices
## Phase 2 of the Experimental Workflow (`results/final_experimental_report.md` - Post-Batch Job)

### Operational Workflow:
1. **Batch Training & Evaluation**: Training of all 3 architectures for 50 epochs with Focal Loss ($\gamma=2.0, \alpha=0.5$) at `seed=0` is executed via batch job:
   ```bash
   bash scripts/launch_batch_training.sh
   # or: bash scripts/run_training.sh --all --gpu --epochs 50
   ```
2. **Role of this Notebook**: Run **after batch job completion** to:
   - Verify training logs and checkpoint integrity (`best_model.pth` under `results/models/`).
   - Load the 3 frozen models and run formal comparative evaluation on the Test Set (`181,366` flows).
   - Compute **Macro F1-Score**, **Balanced Accuracy**, and per-class F1 across all 13 classes.
   - Generate and save the **3 normalized Confusion Matrices (0–100%)** with a shared colorbar for publication.
   - Merge computational efficiency and single-thread edge CPU latency metrics (`results/metrics/edge_latency_profiling.csv`).
   - Export the summary table `results/metrics/benchmark_metrics.csv` for `results/final_experimental_report.md`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
MODELS_DIR = RESULTS_DIR / 'models'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

for d in [MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
from xai.model_loader import load_checkpoint, list_experiments

seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluation device: {device}")

### 2.1 Batch Job Status & Checkpoint Verification (Task T2.3)
Inspect `results/` to verify that the batch job finished training all 3 models.

In [ ]:
TARGET_MODELS = ['Lopez17CNN', 'BiflowTransformer', 'MedDefenderTransformer']
RUN_IDS = {
    'Lopez17CNN': 'S2_R01_Lopez17CNN_multi_10pkts_seed0',
    'BiflowTransformer': 'S2_R02_BiflowTransformer_multi_10pkts_seed0',
    'MedDefenderTransformer': 'S2_R03_MedDefender_multi_10pkts_seed0'
}

print("======================================================================")
print(" BATCH JOB RESULTS CHECK (scripts/run_training.sh)")
print("======================================================================")
all_ready = True
for name, exp_id in RUN_IDS.items():
    best_pth = MODELS_DIR / name / "best_model.pth"
    log_file = RESULTS_DIR / f"train_{exp_id}.log"
    
    status_pth = "✓ Present" if best_pth.exists() else "✗ Missing"
    status_log = "✓ Present" if log_file.exists() else "✗ Missing"
    
    if not best_pth.exists():
        all_ready = False
        
    print(f"[{name:24s}] Checkpoint: {status_pth} | Log: {status_log}")

if not all_ready:
    print("\n[INFO] One or more models have not yet completed training via the Batch Job.")
    print("To launch the batch job from a terminal, run:")
    print("    bash scripts/launch_batch_training.sh")
else:
    print("\n[OK] All 3 models completed training in the Batch Job!")

### 2.2 Test DataLoader Initialization (Task T1.4 & T2.4)
Load the fixed test set at `seed=0` (`181,366` flows) for controlled *ceteris paribus* evaluation.

In [ ]:
from datasets.data_loader import get_loaders

DATA_CANDIDATES = [
    Path('/nas/data/datasets/cic-iomt'),
    REPO_ROOT / 'dataset',
    REPO_ROOT / 'data'
]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), REPO_ROOT / 'dataset')

NUM_PKTS = 10
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
NUM_CLASSES = 13

# Read target classes from classes.txt
CLASSES_FILE = DATA_DIR / 'classes.txt'
if CLASSES_FILE.exists():
    with open(CLASSES_FILE) as f:
        CLASS_NAMES = [l.strip() for l in f if l.strip()]
else:
    CLASS_NAMES = [f"Class_{i}" for i in range(NUM_CLASSES)]

_, _, tst_loader, _ = get_loaders(
    datasets=['cic-iomt'],
    num_tasks=1,
    nc_first_task=NUM_CLASSES,
    nc_incr_tasks=0,
    batch_size=128,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    seed=0,
    num_pkts=NUM_PKTS,
    fields=FIELDS
)
test_loader = tst_loader[0]
print(f"[OK] Test DataLoader loaded: {len(test_loader.dataset):,} total flows.")
assert len(test_loader.dataset) == 181_366, f"Expected 181,366 test flows, found {len(test_loader.dataset):,}" 

### 2.3 Formal Evaluation of the 3 Models on the Test Set (Task T2.4)
If checkpoints exist, compute test predictions and extract macro and per-class performance metrics.

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            outputs = model(x)
            if isinstance(outputs, (list, tuple)):
                outputs = outputs[0]
            preds = outputs.argmax(dim=-1).cpu()
            y_true.extend(y.numpy())
            y_pred.extend(preds.numpy())
    return np.array(y_true), np.array(y_pred)

if not all_ready:
    raise RuntimeError("Missing checkpoints in results/models/: run the training batch job first.")

results = {}
for name, exp_id in RUN_IDS.items():
    print(f"Evaluating: {name}...")
    model, _ = load_checkpoint(exp_name=f"cic-iomt_scratch_{exp_id}", results_dir=str(RESULTS_DIR),
                               network=name, device=str(device))
    y_true, y_pred = evaluate_model(model, test_loader, device)

    macro_f1 = f1_score(y_true, y_pred, average='macro') * 100
    bal_acc = balanced_accuracy_score(y_true, y_pred) * 100
    cm = confusion_matrix(y_true, y_pred, normalize='true') * 100  # expressed as 0-100% percentage

    results[name] = {
        'Macro_F1': macro_f1,
        'Balanced_Acc': bal_acc,
        'cm': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }
    print(f"  -> {name}: Macro-F1 = {macro_f1:.2f}% | Balanced Accuracy = {bal_acc:.2f}%")

### 2.4 Normalized Confusion Matrix Generation (Task T2.6)
Plot the 3 row-normalized Confusion Matrices (0–100%) with a shared colorbar for publication.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=True)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])

for idx, (name, data) in enumerate(results.items()):
    ax = axes[idx]
    cm = data['cm']
    sns.heatmap(
        cm,
        ax=ax,
        cmap='Blues',
        vmin=0,
        vmax=100,
        annot=False,
        cbar=(idx == 2),
        cbar_ax=cbar_ax if (idx == 2) else None,
        square=True
    )
    ax.set_title(f"{name}\n(Macro-F1: {data['Macro_F1']:.1f}% | BA: {data['Balanced_Acc']:.1f}%)", fontsize=12, fontweight='bold')
    ax.set_xlabel(r'Predicted Class, $\hat{y} \in [0..12]$', fontsize=10)
    if idx == 0:
        ax.set_ylabel(r'True Network Class, $y \in [0..12]$', fontsize=10)
cbar_ax.set_ylabel('Recall / Accuracy Rate [%]', fontsize=10)

plt.suptitle('Normalized Confusion Matrix Comparison (%) - CIC-IoMT-2024 (13 Classes, T=10, Seed 0)', fontsize=14, y=1.02)
out_cm_png = FIGURES_DIR / 'confusion_matrices_benchmark.png'
out_cm_pdf = FIGURES_DIR / 'confusion_matrices_benchmark.pdf'
plt.savefig(out_cm_png, dpi=300, bbox_inches='tight')
plt.savefig(out_cm_pdf, bbox_inches='tight')
plt.show()
print(f"[OK] Figures saved to:\n  - {out_cm_png}\n  - {out_cm_pdf}")

### 2.5 Edge CPU Latency Integration & Benchmark Summary Table (Task T2.5)
Combine classification accuracy metrics with single-thread edge CPU inference latency measurements.

In [ ]:
import json
import re

latency_csv = METRICS_DIR / 'edge_latency_profiling.csv'
df_lat = pd.read_csv(latency_csv).set_index('Network')  # generated by scripts/profile_latency.py --all
if df_lat['Checkpoint'].astype(str).str.startswith('None').any():
    raise RuntimeError(f"{latency_csv} was not measured on trained checkpoints: run "
                       "'python scripts/profile_latency.py --all' after the training batch job.")

def train_summary(exp_id):
    """Extract best-model epoch and minimum validation loss from raw_log, and training duration from train log."""
    exp_dir = RESULTS_DIR / f"cic-iomt_scratch_{exp_id}"
    raw_logs = sorted(exp_dir.glob('raw_log-*.txt'))
    if not raw_logs:
        raise FileNotFoundError(f"No raw_log-*.txt found in {exp_dir}")
    with open(raw_logs[-1]) as f:
        valid = [(r['iter'], r['value']) for r in map(json.loads, f)
                 if r['name'] == 'loss' and r['group'] == 'valid']
    best_epoch, val_loss_min = min(valid, key=lambda p: p[1])
    m = re.search(r'TRAINING_TIME_S=(\d+)', (RESULTS_DIR / f"train_{exp_id}.log").read_text())
    if m is None:
        raise RuntimeError(f"TRAINING_TIME_S missing in train_{exp_id}.log: re-run scripts/run_training.sh")
    return best_epoch, val_loss_min, int(m.group(1))

benchmark_records = []
for name, data in results.items():
    lat = df_lat.loc[name]
    best_epoch, val_loss_min, train_s = train_summary(RUN_IDS[name])
    rec = {
        'Run_ID': RUN_IDS[name][:6],
        'Model': name,
        'Macro_F1_%': round(data['Macro_F1'], 2),
        'Balanced_Acc_%': round(data['Balanced_Acc'], 2),
        'Best_Epoch': best_epoch,
        'Val_Loss_Min': round(val_loss_min, 4),
        'Training_Time_min': round(train_s / 60, 1),
        'Params': int(lat['Params']),
        'Size_MB': float(lat['Size_MB']),
        'Latency_CPU_ms': float(lat['Latency_Median_ms']),
        'Latency_P95_ms': float(lat['Latency_P95_ms']),
        'Throughput_fps': float(lat['Throughput_fps'])
    }
    benchmark_records.append(rec)

df_benchmark = pd.DataFrame(benchmark_records)
df_benchmark.to_csv(METRICS_DIR / 'benchmark_metrics.csv', index=False)
print("\n======================================================================")
print(" FINAL COMPARATIVE BENCHMARK TABLE (Phase 2)")
print("======================================================================")
display(df_benchmark) if 'display' in globals() else print(df_benchmark)

### 2.6 Markdown Formatting for `results/final_experimental_report.md`
Format summary Markdown rows ready for the experimental report (`results/final_experimental_report.md`).

In [ ]:
print("### Markdown table for results/final_experimental_report.md (Table 1):")
print("| Run ID | Architecture | Epochs | Macro-F1 (%) | Balanced Acc (%) | Best Epoch | Val Loss Min | Training Time |")
print("| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |")
for r in benchmark_records:
    print(f"| **{r['Run_ID']}** | `{r['Model']}` | 50 | **{r['Macro_F1_%']}%** | **{r['Balanced_Acc_%']}%** | {r['Best_Epoch']} | {r['Val_Loss_Min']} | {r['Training_Time_min']} min |")